In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os

In [2]:
GOES_EAST_PROJ4 = "+proj=geos +lon_0=-75 +h=35786023 +x_0=0 +y_0=0 +sweep=x +datum=WGS84 +units=m +no_defs"
HIMAWARI_PROJ4 = "+proj=geos +lon_0=140.700 +h=35785863 +x_0=0 +y_0=0 +sweep=x +datum=WGS84 +units=m +no_defs"
SEVIRI_PROJ4 = "+proj=geos +lon_0=0 +h=35785831 +x_0=0 +y_0=0 +sweep=x +datum=WGS84 +units=m +no_defs"

In [3]:
goes_path = '/mnt/disks/pretraining/2025-esl-3dclouds-extremes-datasets/pre-training/goes/mcmip'
him_path = '/mnt/disks/pretraining/2025-esl-3dclouds-extremes-datasets/pre-training/himawari/l1b'
msg_path = '/mnt/disks/pretraining/2025-esl-3dclouds-extremes-datasets/pre-training/msg/l1b'

In [4]:
goes_files = sorted([os.path.join(goes_path, f) for f in os.listdir(goes_path) if f.endswith('.nc')])
him_files = sorted([os.path.join(him_path, f) for f in os.listdir(him_path) if f.endswith('.nc')])
msg_files = sorted([os.path.join(msg_path, f) for f in os.listdir(msg_path) if f.endswith('.nc')])

In [5]:
ds = xr.open_dataset(goes_files[0])

In [6]:
ds

<xarray.Dataset> Size: 92MB
Dimensions:      (channel: 16, y: 1024, x: 1024, angle: 2)
Coordinates:
    t            datetime64[ns] 8B ...
  * y            (y) float32 4kB 0.0968 0.09674 0.09668 ... 0.03956 0.03951
  * x            (x) float32 4kB -0.1011 -0.101 -0.1009 ... -0.04382 -0.04376
    y_image      float32 4B ...
    x_image      float32 4B ...
  * channel      (channel) <U7 448B 'CMI_C01' 'CMI_C02' ... 'CMI_C15' 'CMI_C16'
Dimensions without coordinates: angle
Data variables:
    data         (channel, y, x) float32 67MB ...
    latitude     (y, x) float32 4MB ...
    longitude    (y, x) float32 4MB ...
    sat_angle    (angle, y, x) float32 8MB ...
    solar_angle  (angle, y, x) float32 8MB ...
Attributes: (12/29)
    naming_authority:          gov.nesdis.noaa
    Conventions:               CF-1.7
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    standard_name_vocabulary:  CF Standard Name Table (v25, 05 July 2013)
    institution:               DOC/NOAA/NESDIS > U.S. Department of Commerce,...
    project:                   GOES
    ...                        ...
    date_created:              2018-01-02T21:11:24.6Z
    time_coverage_start:       2018-01-02T21:00:39.2Z
    time_coverage_end:         2018-01-02T21:11:15.9Z
    timeline_id:               ABI Mode 3
    production_data_source:    Realtime
    id:                        2618c4db-5f58-49d6-aab1-a36501424b3e

In [7]:
# Add coordinate reference system (CRS) information

ds = ds.rio.write_crs(GOES_EAST_PROJ4, inplace=True)

In [8]:
ds.goes_imager_projection.attrs

{'crs_wkt': 'PROJCS["unknown",GEOGCS["unknown",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Geostationary_Satellite"],PARAMETER["central_meridian",-75],PARAMETER["satellite_height",35786023],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=geos +sweep=x +lon_0=-75 +h=35786023 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"]]',
 'semi_major_axis': 6378137.0,
 'semi_minor_axis': 6356752.314245179,
 'inverse_flattening': 298.257223563,
 'reference_ellipsoid_name': 'WGS 84',
 'longitude_of_prime_meridian': 0.0,
 'prime_meridian_name': 'Greenwich',
 'geographic_crs_name': 'unknown',
 'horizontal_datum_name': 'World Geodetic System 1984',
 'projected_crs_name': 'unknown',
 'grid_mapping_name': 'g

In [9]:
import rioxarray
import xarray as xr
from rasterio.enums import Resampling
from typing import Tuple

rioxarray_samplers = {
    "bilinear": Resampling.bilinear,
    "cubic": Resampling.cubic,
    "cubic_spline": Resampling.cubic_spline,
    "nearest": Resampling.nearest,
}

def resample_rioxarray(ds: xr.Dataset, resolution: Tuple[int, int]=(1_000, 1_000), method: str="bilinear") -> xr.Dataset:
    """
    Resamples a raster dataset using rasterio-xarray.

    Parameters:
        ds (xr.Dataset): The input dataset to be resampled.
        resolution (int): The desired resolution of the resampled dataset. Default is 1_000.
        method (str): The resampling method to be used. Default is "bilinear".

    Returns:
        xr.Dataset: The resampled dataset.
    """

    ds = ds.rio.reproject(
        ds.rio.crs,
        resolution=resolution,
        resample=rioxarray_samplers[method], 
    )
    return ds

In [10]:
ds.rio.resolution()

(5.599999853126697e-05, -5.599999853126697e-05)

In [11]:
# Check coordinate information
print("X coordinate info:")
print(f"X coordinate name: {ds.dims}")
print(f"X coordinate values (first 5): {ds.x.values[:5]}")
print(f"X coordinate range: {ds.x.values.min()} to {ds.x.values.max()}")
print(f"X coordinate units: {ds.x.attrs.get('units', 'No units specified')}")
print()
print("Y coordinate info:")
print(f"Y coordinate values (first 5): {ds.y.values[:5]}")
print(f"Y coordinate range: {ds.y.values.min()} to {ds.y.values.max()}")
print(f"Y coordinate units: {ds.y.attrs.get('units', 'No units specified')}")
print()
print("Current CRS:", ds.rio.crs)

X coordinate info:
X coordinate name: FrozenMappingWarningOnValuesAccess({'channel': 16, 'y': 1024, 'x': 1024, 'angle': 2})
X coordinate values (first 5): [-0.10105199 -0.100996   -0.10093999 -0.10088399 -0.10082799]
X coordinate range: -0.10105199366807938 to -0.04376399517059326
X coordinate units: rad

Y coordinate info:
Y coordinate values (first 5): [0.09679599 0.09673999 0.09668399 0.096628   0.096572  ]
Y coordinate range: 0.039507992565631866 to 0.09679599106311798
Y coordinate units: rad

Current CRS: PROJCS["unknown",GEOGCS["unknown",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Geostationary_Satellite"],PARAMETER["central_meridian",-75],PARAMETER["satellite_height",35786023],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AX

In [12]:
# Convert coordinates from radians to meters for geostationary projection
# For GOES, the satellite height is 35,786,023 meters (from the CRS info)

# Get satellite height from the projection info
satellite_height = 35786023  # meters

# Convert x and y coordinates from radians to meters
x_meters = ds.x.values * satellite_height
y_meters = ds.y.values * satellite_height

print(f"Original X range (rad): {ds.x.values.min():.6f} to {ds.x.values.max():.6f}")
print(f"Converted X range (m): {x_meters.min():.0f} to {x_meters.max():.0f}")
print(f"Original Y range (rad): {ds.y.values.min():.6f} to {ds.y.values.max():.6f}")
print(f"Converted Y range (m): {y_meters.min():.0f} to {y_meters.max():.0f}")

# Update the dataset coordinates
ds_corrected = ds.assign_coords(x=x_meters, y=y_meters)

# Update the coordinate attributes to reflect meters
ds_corrected.x.attrs['units'] = 'm'
ds_corrected.y.attrs['units'] = 'm'

print("\nAfter coordinate conversion:")
print(f"New resolution: {ds_corrected.rio.resolution()}")
print(f"Resolution in km: {abs(ds_corrected.rio.resolution()[0]/1000):.1f} km")

Original X range (rad): -0.101052 to -0.043764
Converted X range (m): -3616249 to -1566139
Original Y range (rad): 0.039508 to 0.096796
Converted Y range (m): 1413834 to 3463944

After coordinate conversion:
New resolution: (2004.017228739003, -2004.0173509286412)
Resolution in km: 2.0 km


In [13]:
# Now let's test resampling with the corrected coordinates
# Resample to 1km resolution
ds_resampled = resample_rioxarray(ds_corrected, resolution=(1000, 1000), method="bilinear")

print("Original dataset:")
print(f"Resolution: {ds_corrected.rio.resolution()}")
print(f"Shape: {ds_corrected.data.shape}")
print()
print("Resampled dataset:")
print(f"Resolution: {ds_resampled.rio.resolution()}")
print(f"Shape: {ds_resampled.data.shape}")
print(f"Resolution in km: {abs(ds_resampled.rio.resolution()[0]/1000):.1f} km")

Original dataset:
Resolution: (2004.017228739003, -2004.0173509286412)
Shape: (16, 1024, 1024)

Resampled dataset:
Resolution: (1000.0, -1000.0)
Shape: (16, 2053, 2053)
Resolution in km: 1.0 km


## Resampling Geostationary Satellite Data

The dataset coordinates were in **radians** (the satellite's native viewing angle coordinates) rather than **meters** (the projected coordinate system). This is common with geostationary satellite data like GOES, Himawari, and MSG.

### The Fix:
1. **Convert coordinates from radians to meters** by multiplying by the satellite height
2. **Update coordinate attributes** to reflect the correct units
3. **Resample using the corrected coordinates**

### Complete workflow:

In [14]:
def convert_coordinates(ds: xr.Dataset, satellite_type: str = "goes") -> xr.Dataset:
    """
    Convert satellite coordinates from radians to meters for geostationary projections.
    
    Parameters:
        ds (xr.Dataset): Input dataset with coordinates in radians
        satellite_type (str): Type of satellite ("goes", "himawari", "msg")
        
    Returns:
        xr.Dataset: Dataset with corrected coordinates in meters
    """
    # Satellite heights in meters
    satellite_heights = {
        "goes": 35786023,      # GOES-16/17
        "himawari": 35785863,  # Himawari-8/9  
        "msg": 35785831        # MSG/SEVIRI
    }
    
    if satellite_type.lower() not in satellite_heights:
        raise ValueError(f"Unknown satellite type: {satellite_type}")
    
    satellite_height = satellite_heights[satellite_type.lower()]
    
    # Check if coordinates are in radians
    x_units = ds.x.attrs.get('units', '')
    y_units = ds.y.attrs.get('units', '')
    
    if x_units == 'rad' and y_units == 'rad':
        print(f"Converting {satellite_type.upper()} coordinates from radians to meters...")
        
        # Convert coordinates
        x_meters = ds.x.values * satellite_height
        y_meters = ds.y.values * satellite_height
        
        # Update dataset
        ds_corrected = ds.assign_coords(x=x_meters, y=y_meters)
        ds_corrected.x.attrs['units'] = 'm'
        ds_corrected.y.attrs['units'] = 'm'
        
        print(f"Original resolution: {ds.rio.resolution()}")
        print(f"Corrected resolution: {ds_corrected.rio.resolution()}")
        print(f"Resolution in km: {abs(ds_corrected.rio.resolution()[0]/1000):.1f} km")
        
        return ds_corrected
    else:
        print("Coordinates are already in proper units, no conversion needed.")
        return ds

In [15]:
# Complete workflow example:

# 1. Load your dataset
ds_original = xr.open_dataset(goes_files[0])
ds_original = ds_original.rio.write_crs(GOES_EAST_PROJ4, inplace=True)

print("Step 1 - Original dataset:")
print(f"Resolution: {ds_original.rio.resolution()} (in radians - WRONG!)")
print()

# 2. Fix the coordinates 
ds_corrected = convert_coordinates(ds_original, satellite_type="goes")
print()

# 3. Now resample to your desired resolution
ds_resampled_1km = resample_rioxarray(ds_corrected, resolution=(1000, 1000), method="bilinear")
ds_resampled_5km = resample_rioxarray(ds_corrected, resolution=(5000, 5000), method="bilinear")
ds_resampled_2km = resample_rioxarray(ds_corrected, resolution=(2000, 2000), method="bilinear")
ds_resampled_3km = resample_rioxarray(ds_corrected, resolution=(3000.403, 3000.403), method="bilinear")

print("Step 3 - Resampling results:")
print(f"Original: {ds_corrected.data.shape} at {abs(ds_corrected.rio.resolution()[0]/1000):.1f} km")
# print(f"1km: {ds_resampled_1km.data.shape} at {abs(ds_resampled_1km.rio.resolution()[0]/1000):.1f} km") 
# print(f"5km: {ds_resampled_5km.data.shape} at {abs(ds_resampled_5km.rio.resolution()[0]/1000):.1f} km")
print(f"2km: {ds_resampled_2km.data.shape} at {abs(ds_resampled_2km.rio.resolution()[0]/1000):.1f} km")
print(f"3km: {ds_resampled_3km.data.shape} at {abs(ds_resampled_3km.rio.resolution()[0]/1000):.1f} km")

Step 1 - Original dataset:
Resolution: (5.599999853126697e-05, -5.599999853126697e-05) (in radians - WRONG!)

Converting GOES coordinates from radians to meters...
Original resolution: (5.599999853126697e-05, -5.599999853126697e-05)
Corrected resolution: (2004.017228739003, -2004.0173509286412)
Resolution in km: 2.0 km

Step 3 - Resampling results:
Original: (16, 1024, 1024) at 2.0 km
2km: (16, 1027, 1027) at 2.0 km
3km: (16, 684, 684) at 3.0 km


In [16]:
# Complete workflow example:

# 1. Load your dataset
ds_original = xr.open_dataset(him_files[0])
ds_original = ds_original.rio.write_crs(HIMAWARI_PROJ4, inplace=True)

print("Step 1 - Original dataset:")
print(f"Resolution: {ds_original.rio.resolution()}")
print()

# 2. Fix the coordinates 
ds_corrected = convert_coordinates(ds_original, satellite_type="himawari")
print()

# 3. Now resample to your desired resolution
ds_resampled_1km = resample_rioxarray(ds_corrected, resolution=(1000, 1000), method="bilinear")
ds_resampled_5km = resample_rioxarray(ds_corrected, resolution=(5000, 5000), method="bilinear")
ds_resampled_2km = resample_rioxarray(ds_corrected, resolution=(2004, 2004), method="bilinear")
ds_resampled_3km = resample_rioxarray(ds_corrected, resolution=(3000.403, 3000.403), method="bilinear")

print("Step 3 - Resampling results:")
print(f"Original: {ds_corrected.data.shape} at {abs(ds_corrected.rio.resolution()[0]/1000):.1f} km")
# print(f"1km: {ds_resampled_1km.data.shape} at {abs(ds_resampled_1km.rio.resolution()[0]/1000):.1f} km") 
# print(f"5km: {ds_resampled_5km.data.shape} at {abs(ds_resampled_5km.rio.resolution()[0]/1000):.1f} km")
print(f"2km: {ds_resampled_2km.data.shape} at {abs(ds_resampled_2km.rio.resolution()[0]/1000):.1f} km")
print(f"3km: {ds_resampled_3km.data.shape} at {abs(ds_resampled_3km.rio.resolution()[0]/1000):.1f} km")

Step 1 - Original dataset:
Resolution: (2000.0, -2000.0)

Coordinates are already in proper units, no conversion needed.

Step 3 - Resampling results:
Original: (16, 1024, 1024) at 2.0 km
2km: (16, 1022, 1022) at 2.0 km
3km: (16, 683, 683) at 3.0 km


In [17]:
# Complete workflow example:

# 1. Load your dataset
ds_original = xr.open_dataset(msg_files[0])
ds_original = ds_original.rio.write_crs(SEVIRI_PROJ4, inplace=True)

print("Step 1 - Original dataset:")
print(f"Resolution: {ds_original.rio.resolution()}")
print()

# 2. Fix the coordinates 
ds_corrected = convert_coordinates(ds_original, satellite_type="msg")
print()

# 3. Now resample to your desired resolution
ds_resampled_1km = resample_rioxarray(ds_corrected, resolution=(1000, 1000), method="bilinear")
ds_resampled_5km = resample_rioxarray(ds_corrected, resolution=(5000, 5000), method="bilinear")
ds_resampled_2km = resample_rioxarray(ds_corrected, resolution=(2004, 2004), method="bilinear")
ds_resampled_3km = resample_rioxarray(ds_corrected, resolution=(3000.403, 3000.403), method="bilinear")

print("Step 3 - Resampling results:")
print(f"Original: {ds_corrected.data.shape} at {abs(ds_corrected.rio.resolution()[0]/1000):.1f} km")
# print(f"1km: {ds_resampled_1km.data.shape} at {abs(ds_resampled_1km.rio.resolution()[0]/1000):.1f} km") 
# print(f"5km: {ds_resampled_5km.data.shape} at {abs(ds_resampled_5km.rio.resolution()[0]/1000):.1f} km")
print(f"2km: {ds_resampled_2km.data.shape} at {abs(ds_resampled_2km.rio.resolution()[0]/1000):.1f} km")
print(f"3km: {ds_resampled_3km.data.shape} at {abs(ds_resampled_3km.rio.resolution()[0]/1000):.1f} km")

Step 1 - Original dataset:
Resolution: (-3000.4032258064517, 3000.4031036168135)

Coordinates are already in proper units, no conversion needed.

Step 3 - Resampling results:
Original: (11, 1024, 1024) at 3.0 km
2km: (11, 1534, 1534) at 2.0 km
3km: (11, 1025, 1025) at 3.0 km


## Adding resampling to the load function

In [18]:
# GOES wavelengths in nanometers
GOES_WAVELENGTHS = {
    "CMI_C01": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 450.5,
        "center_wavelength": 470.0,
        "max_wavelength": 490.6,
    },  # 0.47,
    "CMI_C02": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 596.3,
        "center_wavelength": 640.0,
        "max_wavelength": 682.1,
    },  # 0.64,
    "CMI_C03": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 846.9,
        "center_wavelength": 870.0,
        "max_wavelength": 882.0,
    },  # 0.87,
    "CMI_C04": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 1366.3,
        "center_wavelength": 1380.0,
        "max_wavelength": 1380.3,
    },  # 1.38,
    "CMI_C05": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 1587.6,
        "center_wavelength": 1610.0,
        "max_wavelength": 1632.4,
    },  # 1.61,
    "CMI_C06": {
        "reso_og": 3000,
        "band_type": "TOA Reflectance",
        "min_wavelength": 2220.2,
        "center_wavelength": 2250.0,
        "max_wavelength": 2265.5,
    },  # 2.25,
    "CMI_C07": {
        "reso_og": 3000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 3802.7,
        "center_wavelength": 3890.0,
        "max_wavelength": 3992.2,
    },  # 3.89,
    "CMI_C08": {
        "reso_og": 3000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 5790.4,
        "center_wavelength": 6170.0,
        "max_wavelength": 6590.7,
    },  # 6.17,
    "CMI_C09": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 6725.0,
        "center_wavelength": 6930.0,
        "max_wavelength": 7142.9,
    },  # 6.93,
    "CMI_C10": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 7242.7,
        "center_wavelength": 7340.0,
        "max_wavelength": 7431.1,
    },  # 7.34,
    "CMI_C11": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 8226.4,
        "center_wavelength": 8440.0,
        "max_wavelength": 8663.3,
    },  # 8.44,
    "CMI_C12": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 9423.3,
        "center_wavelength": 9610.0,
        "max_wavelength": 9800.1,
    },  # 9.61,
    "CMI_C13": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 10177.1,
        "center_wavelength": 10330.0,
        "max_wavelength": 10481.1,
    },  # 10.33,
    "CMI_C14": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 10815.5,
        "center_wavelength": 11190.0,
        "max_wavelength": 11603.6,
    },  # 11.19,
    "CMI_C15": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 11825.9,
        "center_wavelength": 12270.0,
        "max_wavelength": 12747.0,
    },  # 12.27,
    "CMI_C16": {
        "reso_og": 2000,
        "band_type": "TOA Normalised Brightness Temperature",
        "min_wavelength": 12990.4,
        "center_wavelength": 13270.0,
        "max_wavelength": 13559.3,
    },  # 13.27,
}

In [19]:
from __future__ import annotations

import autoroot  # required for imports from src
import numpy as np
import xarray as xr

def scale_reflectance(data_dict):
    """
    Scale the reflectance data to the range [0, 100].
    """
    for i, band_name in enumerate(data_dict["band_names"]):
        if "Reflectance" in data_dict["sensor_info"][band_name]["band_type"]:
            # Scale reflectance data to [0, 100] range
            data_dict["data"][i] = data_dict["data"][i] * 100
    return data_dict


def load_goes_file(
    file: str,
    load_zenith: bool = True,
    load_solar: bool = True,
    patch_size: list
    | None = None,  # Whether to crop the data to a smaller patch size (e.g. [128, 128] for pre-training
    resolution: float = None,  # Desired resolution in meters (e.g. 2000 for 2km)
    center_crop: bool = False,  # If True, will crop to the center of the image
    radius: int = 0,  # Radius for cropping, if center_crop is True
):
    if not file.endswith(".nc"):
        raise NotImplementedError("Unsupported file format.")

    # define an empty dictionary
    data_dict = {}
    # open file
    with xr.open_dataset(file) as ds:

        if resolution is not None:
            ds = ds.rio.write_crs(GOES_EAST_PROJ4, inplace=True)
            ds = convert_coordinates(ds, satellite_type="goes")
            ds = resample_rioxarray(ds, resolution=(resolution, resolution), method="bilinear")
            
        # extract data
        if "data" in ds.data_vars:
            data_dict["data"] = ds.data.values.astype(np.float32)
        else:
            data_dict["data"] = (
                ds[list(GOES_WAVELENGTHS.keys())].to_array().values.astype(np.float32)
            )

        # extract coordinates
        # calculate latitude and longitude coordinates
        # Fix lat/lon encoding bug
        if "latitude" in ds.data_vars and "longitude" in ds.data_vars:
            lat_offset = (
                (
                    ds.latitude.encoding["scale_factor"]
                    + ds.latitude.encoding["add_offset"]
                )
                * 2
                if ds.latitude.encoding["add_offset"] > 0
                else 0
            )
            lon_offset = (
                (
                    ds.longitude.encoding["scale_factor"]
                    + ds.longitude.encoding["add_offset"]
                )
                * 2
                if ds.longitude.encoding["add_offset"] > 0
                else 0
            )
            latitudes = (ds.latitude - lat_offset).fillna(
                ds.latitude.encoding["add_offset"]
            )
            longitudes = (ds.longitude - lon_offset).fillna(
                ds.longitude.encoding["add_offset"]
            )

            data_dict["coords"] = np.stack(
                [latitudes.values, longitudes.values], axis=0
            )

        # add band names and wavelengths
        data_dict["band_names"] = list(GOES_WAVELENGTHS.keys())
        data_dict["wavelengths"] = [
            val.get("center_wavelength") for val in GOES_WAVELENGTHS.values()
        ]
        data_dict["sensor_info"] = GOES_WAVELENGTHS

        if "data" not in ds.data_vars:
            # Scale reflectance data to [0, 100] range
            data_dict = scale_reflectance(data_dict)

        # calculate the zenith angle
        if load_zenith:
            if "sat_angle" in ds.data_vars:
                data_dict["sat_angle"] = ds.sat_angle.values
        if load_solar:
            if "solar_angle" in ds.data_vars:
                data_dict["solar_angle"] = ds.solar_angle.values

    return data_dict

In [20]:
ds = load_goes_file(goes_files[15], resolution=4000)

Converting GOES coordinates from radians to meters...
Original resolution: (5.5999991248196986e-05, -5.599999853126697e-05)
Corrected resolution: (2004.0171065493646, -2004.0171065493646)
Resolution in km: 2.0 km


In [21]:
ds['solar_angle'].shape

(2, 514, 514)